<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# 第 3 章：编写注意力机制

本 notebook 中使用的包：

In [1]:
from importlib.metadata import version

print("torch version:", version("torch"))

torch version: 2.4.0


- 本章涵盖注意力机制（attention mechanisms），它是 LLM 的核心引擎：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/01.webp?123" width="500px">

**Figure 3.1 — Chapter 3 在整个 LLM 构建流程中的位置**

```
Chapter 2: 数据准备          Chapter 3: 注意力机制         Chapter 4: GPT 模型
─────────────────          ──────────────────          ─────────────────
原始文本                    input_embeddings              完整 GPT
  ↓ 分词                      (8, 4, 256)                    ↑
  ↓ 滑动窗口                       ↓                        ↓
  ↓ Embedding              ┌──────────────────┐        Transformer Block
                           │  自注意力         │        ├─ Multi-Head Attention ← Ch3
input_embeddings           │  因果注意力       │        ├─ Feed Forward
(8, 4, 256)                │  多头注意力       │        ├─ Layer Norm
                           └──────────────────┘        └─ Dropout
                                ↓
                          本章核心：让模型学会
                          「关注输入中最重要的部分」
```

> **注意力机制是 LLM 的核心引擎**：它决定了模型在处理当前 token 时，应该「看」输入序列中的哪些其他 token。本章从最简单的自注意力开始，逐步构建到完整的多头注意力。

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/02.webp" width="600px">

**Figure 3.2 — Chapter 3 学习路线图（由简到繁）**

```
3.3 简单自注意力（无可训练权重）
     │  输入向量之间做点积 → 注意力分数 → 加权求和
     │  问题：没有可学习的参数，模型无法优化
     ↓
3.4 自注意力（带可训练权重 Q/K/V）
     │  引入 Wq, Wk, Wv 三个权重矩阵
     │  Query x Key → 注意力分数 → x Value → 上下文向量
     ↓
3.5 因果注意力（Causal Attention）
     │  加入三角 mask，防止看到未来 token
     │  加入 dropout 正则化
     ↓
3.6 多头注意力（Multi-Head Attention）
     │  并行运行多个注意力头
     │  每个头关注不同的模式/子空间
     ↓
   这就是 GPT 用的注意力机制 ✓
```

| 阶段 | 核心改进 | 新增概念 |
|---|---|---|
| 3.3 简单自注意力 | 向量点积计算相关性 | attention score, softmax, context vector |
| 3.4 可训练自注意力 | 加入 Q/K/V 权重 | Query, Key, Value, 可学习参数 |
| 3.5 因果注意力 | mask + dropout | 因果 mask, 上三角归零, dropout |
| 3.6 多头注意力 | 并行多组 Q/K/V | num_heads, head_dim, weight split |

&nbsp;
## 3.1 建模长序列的问题

> **为什么需要注意力机制？**
>
> 在 Transformer 出现之前，序列建模主要依赖 RNN/LSTM。这类模型按时间步顺序处理输入，
> 并用一个**固定长度**的隐藏状态（hidden state）来「压缩」整段输入序列。这带来两个根本缺陷：
>
> 1. **长距离依赖难以捕获**：当序列变长时，早期 token 的信息在反复传递中逐渐衰减，
>    模型很难把句子开头的主语与结尾的谓语联系起来。
> 2. **信息瓶颈**：无论输入多长，最终都被压缩进一个固定维度的向量，容易丢失细节。
>
> 注意力机制（attention）正是为解决这两个问题而提出的：它让模型在每一步都能「直接看到」
> 输入序列中的**所有**位置，并按需分配不同的关注权重，从而打破顺序处理的瓶颈。

- 本节没有代码
- 由于源语言和目标语言的语法结构存在差异，逐词翻译文本是不可行的：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/03.webp" width="400px">

**Figure 3.3 — 逐词翻译的问题（为什么需要注意力）**

```
源语言（英语）:   The cat sat on the mat because it was tired
                  ↓ 逐词翻译 ↓
目标语言（中文）:  猫 坐 垫子 上 因为 它 累了
                  ↑                ↑
                  语法完全错！       "it" 指代谁？需要看上下文
```

逐词翻译的核心问题：

| 问题 | 说明 | 例子 |
|---|---|---|
| 语序不同 | 不同语言语法结构差异大 | 英语 SVO vs 日语 SOV |
| 一词多义 | 同一个词在不同语境含义不同 | "bank" = 河岸 / 银行 |
| 长距离依赖 | 当前词的含义依赖远处某个词 | "it" 指代前面的 "cat" |
| 上下文必要 | 翻译当前词必须看到整句话 | 不能孤立地逐词处理 |

> **结论**：RNN/LSTM 逐词处理，隐藏状态是唯一的信息载体。序列越长，早期信息丢失越严重（梯度消失）。注意力机制让模型直接「跳转」到输入中相关的位置，彻底解决了这个问题。

- 在 Transformer 模型出现之前，编码器-解码器 RNN 通常被用于机器翻译任务
- 在这种架构中，编码器（encoder）处理源语言的一系列 token，利用隐藏状态（hidden state）——神经网络中的一种中间层——来生成整个输入序列的紧凑表示（a condensed representation）：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/04.webp" width="500px">

**Figure 3.4 — 编码器-解码器 RNN 架构（Transformer 之前的主流方法）**

```
编码器 (Encoder)                              解码器 (Decoder)
逐词处理源语言                                 逐词生成目标语言

  "The"                                          "El"
    ↓                                             ↑
  [h1] ──→ [h2] ──→ [h3] ──→ [h4] ──→ [h_final] ──→ [s1] ──→ [s2] ──→ [s3]
    ↓        ↓        ↓        ↓                   ↑        ↑        ↑
  "cat"   "sat"    "on"    "mat"                 "gato"  "se"   "sentó"

                                    ↑
                          唯一的信息桥梁 = h_final（最后一个隐藏状态）
                          整个输入序列被压缩成一个固定大小的向量
                          → 信息瓶颈！长序列时早期信息丢失严重
```

| | RNN 编码器-解码器 | Transformer（注意力机制） |
|---|---|---|
| **信息传递** | 仅通过最后一个隐藏状态 h_final | 解码器可直接访问所有编码器隐藏状态 |
| **瓶颈** | 严重（所有信息压缩到一个向量） | 无（每步可选择性地关注任意位置） |
| **长序列** | 早期信息遗忘（梯度消失） | 可直接「跳转」到任意位置 |
| **并行性** | 必须按顺序处理（串行） | 可并行处理所有位置 |

> **RNN 的核心缺陷**：编码器必须把整句话「压缩」进一个固定大小的 h_final 向量。句子越长，信息丢失越多。注意力机制打破了这一瓶颈 — 解码器在生成每个词时，可以直接「看回」编码器的所有中间状态，按需提取信息。

&nbsp;
## 3.2 用注意力机制捕获数据依赖关系

> **从 RNN + attention 到 self-attention 的演进**
>
> - 最早，注意力被作为 RNN 编码器-解码器的**附加组件**：解码器在生成每个词时，
>   不再只依赖一个固定向量，而是对所有编码器输出做加权求和，从而「挑选」相关的输入 token。
> - 这种「编码器-解码器注意力」虽然缓解了长距离依赖问题，但 RNN 的顺序计算依旧很慢。
> - Transformer 的关键创新是 **self-attention（自注意力）**：彻底抛弃 RNN 的循环结构，
>   让序列中的每个位置直接与同一序列中的所有位置交互。这样既并行又高效，
>   成为 GPT、BERT 等现代 LLM 的基石。

- 本节没有代码
- 通过注意力机制，网络中负责生成文本的解码器部分能够**选择性**地访问所有输入 token，这意味着在生成某个特定的输出 token 时，某些输入 token 比其他 token 更重要：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/05.webp" width="500px">

**Figure 3.5 — 注意力让解码器选择性访问输入 token**

```
编码器输出（所有 token 的隐藏状态都可以访问）

  h1("The")    h2("cat")    h3("sat")    h4("on")    h5("the")    h6("mat")
    ↑            ↑                         ↑                         ↑
    │            │                         │                         │
    │  权重小     │  权重大                  │  权重中等                │  权重大
    │  (0.05)    │  (0.60)                 │  (0.10)                 │  (0.25)
    │            │                         │                         │
    └────────────┴─────────────────────────┴─────────────────────────┘
                              ↑
                    解码器当前生成 "gato"（猫）
                    
                    不再只用 h_final，而是加权求和所有 token:
                    context = 0.05xh1 + 0.60xh2 + 0.10xh4 + 0.25xh6
                    
                    "cat" 权重最高 → 模型学会了「猫」最相关的是 "cat"
```

| | RNN（无注意力） | RNN + 注意力 |
|---|---|---|
| **解码器看到什么** | 仅 h_final（一个向量） | 所有编码器隐藏状态的加权和 |
| **能否选择** | 不能，信息已压缩 | 能，通过权重决定关注哪些 token |
| **长序列** | 早期信息遗忘 | 可直接访问任意位置的 token |

> **注意力的核心思想**：不再用一个向量「压缩」整句话，而是让解码器在每个生成步骤中，动态地决定「现在最该看输入中的哪些词」，并为每个词分配不同的权重。

- Transformer 中的 self-attention（自注意力）是一种旨在增强输入表示的技术：它让序列中的每个位置都能与同一序列中的所有其他位置进行交互，并据此判断每个其他位置对当前位置的 relevance（相关性）。

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/06.webp" width="300px">

**Figure 3.6 — 从编码器-解码器注意力到自注意力（Self-Attention）的演进**

```
编码器-解码器注意力（翻译任务专用）     自注意力（Self-Attention，通用）

  源语言 token                              同一序列内的 token
  ┌─────────────┐                          ┌─────────────────┐
  │ The cat sat │                          │ The cat sat on │
  └──────┬──────┘                          └────────┬────────┘
         ↓                                          ↓
  编码器隐藏状态                              输入嵌入向量
  h1   h2   h3                              x1   x2   x3   x4
         ↓                                          ↓
  解码器查询                                每个位置同时做 Query 和 Key 和 Value
  "El" → ?                                  x1 → 问所有 token（包括自己）：谁和我最相关？
         ↓                                          ↓
  跨语言注意力                               同序列内自交互
  （源 ↔ 目标）                             （自己 ↔ 自己）
```

| | 编码器-解码器注意力 | 自注意力 |
|---|---|---|
| **交互范围** | 源语言 ↔ 目标语言（跨序列） | 同一序列内部（自交互） |
| **用途** | 翻译专用 | 通用（理解、生成、分类...） |
| **Query 来源** | 解码器 | 序列自身 |
| **Key/Value 来源** | 编码器 | 序列自身 |

> **Self-Attention 的突破**：不再需要两个独立的序列（编码器/解码器）。单个序列中的每个 token 都可以「看到」同序列中的所有其他 token，并决定哪些最相关。这让注意力机制从翻译工具变成了通用的序列理解引擎。

&nbsp;
## 3.3 用自注意力关注输入的不同部分

&nbsp;
### 3.3.1 没有可训练权重的简单自注意力机制

- 本节讲解 self-attention 的一个非常简化的版本，它不包含任何可训练权重
- 这纯粹是为了说明目的，**并不是** Transformer 中实际使用的注意力机制
- 下一节（3.3.2）将把这个简单的注意力机制扩展为真正的 self-attention 机制
- 假设给定一个输入序列 $x^{(1)}$ 到 $x^{(T)}$
  - 输入是一段文本（例如 "Your journey starts with one step" 这样的句子），已经按照第 2 章所述转换为 token embedding
  - 例如，$x^{(1)}$ 是一个表示 "Your" 这个词的 d 维向量，以此类推
- **目标：** 为每个输入序列元素 $x^{(i)}$（$x^{(1)}$ 到 $x^{(T)}$）计算上下文向量 $z^{(i)}$（其中 $z$ 和 $x$ 维度相同）
    - 上下文向量 $z^{(i)}$ 是所有输入 $x^{(1)}$ 到 $x^{(T)}$ 的加权和
    - 该上下文向量是针对某个特定输入的「上下文」表示
      - 不用 $x^{(i)}$ 作为任意输入 token 的占位符，我们考虑第二个输入 $x^{(2)}$
      - 为继续用一个具体的例子，我们不用占位符 $z^{(i)}$，而考虑第二个输出上下文向量 $z^{(2)}$
      - 第二个上下文向量 $z^{(2)}$ 是所有输入 $x^{(1)}$ 到 $x^{(T)}$ 的加权和，加权依据是第二个输入元素 $x^{(2)}$
      - attention weights（注意力权重）决定了在计算 $z^{(2)}$ 时，每个输入元素对加权和的贡献大小
      - 简而言之，可以把 $z^{(2)}$ 理解为 $x^{(2)}$ 的一个「增强版」，它还融合了与当前任务相关的所有其他输入元素的信息

> **直观理解注意力计算的三步流程**
>
> ```
>  输入 x^(1)...x^(T)              以 x^(2) 为查询 (query)
>        │
>        ▼
>  ① 点积: ω_ij = x^(i) · x^(j)   → 得到未归一化的 attention scores
>        │
>        ▼
>  ② softmax 归一化               → 得到和为 1 的 attention weights α
>        │
>        ▼
>  ③ 加权求和: z^(2) = Σ α_i·x^(i) → 得到上下文向量
> ```
>
> 第一步用**点积（dot product）**衡量两个向量的相似度——点积越大，说明两个 token 越相关；
> 第二步用 softmax 把得分压成概率分布（和为 1）；第三步按这些概率对所有输入做加权平均，
> 得到融合了全局信息的上下文向量。

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/07.webp" width="400px">

- （请注意，本图中的数字被截断到小数点后一位以减少视觉杂乱；类似地，其他图中的数值也可能是截断后的）

**Figure 3.7 — 第一步：计算注意力分数（attention scores）**

```
输入序列: 3 个 token，每个 3 维嵌入向量

  x1 = [0.43, 0.15, 0.89]   ← "your"
  x2 = [0.55, 0.87, 0.66]   ← "journey"
  x3 = [0.57, 0.85, 0.64]   ← "starts"

为 x2("journey") 计算它与所有 token 的注意力分数:

  score_21 = x2 · x1 = 0.95   ← 与 "your"
  score_22 = x2 · x2 = 1.55   ← 与自己（通常最高）
  score_23 = x2 · x3 = 1.44   ← 与 "starts"

          x1      x2      x3
  x2  [  0.95,   1.55,   1.44 ]   ← 未归一化的 attention scores
```

| 术语 | 含义 | 是否归一化 |
|---|---|---|
| **attention scores** | 原始点积值 | 否（和不为 1） |
| **attention weights** | softmax 归一化后 | 是（和为 1） |

> **点积的直觉**：两个向量的点积衡量它们的「相似度」。方向越一致（越相关），点积越大。x₂ 与自己的点积最大，因为完全重合。

- 按照惯例，未归一化的注意力得分被称为 **"attention scores"（注意力分数）**，而归一化后（和为 1）的注意力分数被称为 **"attention weights"（注意力权重）**

- 下面的代码逐步演示了上图的过程

<br>

- **第 1 步：** 计算未归一化的 attention scores $\omega$
- 假设我们用第二个输入 token 作为查询（query），即 $q^{(2)} = x^{(2)}$，通过点积计算未归一化的 attention scores：
    - $\omega_{21} = x^{(1)} q^{(2)\top}$
    - $\omega_{22} = x^{(2)} q^{(2)\top}$
    - $\omega_{23} = x^{(3)} q^{(2)\top}$
    - ...
    - $\omega_{2T} = x^{(T)} q^{(2)\top}$
- 上式中，$\omega$ 是希腊字母「omega」，用来表示未归一化的 attention scores
    - $\omega_{21}$ 中的下标「21」表示用输入序列元素 2 作为查询，去与输入序列元素 1 做计算

- 假设我们有如下输入句子，它已经按照第 3 章所述被嵌入为 3 维向量（这里使用很小的 embedding 维度仅为了演示方便，使其能在页面内显示而不换行）：

In [2]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

- （在本书中，我们遵循机器学习和深度学习的常见约定：训练样本表示为行，特征值表示为列；在上面展示的 tensor 中，每一行代表一个词，每一列代表一个 embedding 维度）

- 本节的主要目标是演示如何使用第二个输入序列 $x^{(2)}$ 作为查询来计算上下文向量 $z^{(2)}$

- 下图描述了这个过程的第一步：通过点积运算计算 $x^{(2)}$ 与所有其他输入元素之间的 attention scores ω

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/08.webp" width="400px">

**Figure 3.8 — 第一步代码实现：用 x(2) 作为查询计算 attention scores**

```
输入嵌入矩阵 inputs (shape: 6x3, 6 个 token x 3 维)

         dim_0  dim_1  dim_2
  x(1)  [ 0.43,  0.15,  0.89]   ← "your"
  x(2)  [ 0.55,  0.87,  0.66]   ← "journey"  ← 当前查询
  x(3)  [ 0.57,  0.85,  0.64]   ← "starts"
  x(4)  [ 0.22,  0.58,  0.33]   ← "with"
  x(5)  [ 0.77,  0.25,  0.10]   ← "one"
  x(6)  [ 0.05,  0.80,  0.55]   ← "step"

代码: query = inputs[1]              # 取 x(2) 作为查询
      attn_scores_2 = inputs @ query # x(2) 与所有 token 做点积

结果: attn_scores_2 = [0.95, 1.55, 1.44, 0.75, 0.64, 1.00]
                       ↑                       ↑
                       x(1) "your"            x(6) "step"
                       相关性较低               相关性中等
                                      ↑
                                      x(2) 自己最高 (1.55)
```

> **关键操作**：`inputs @ query` 是矩阵乘法 — 6 行各与查询向量做点积，一次性得到 6 个 attention scores。无需 for 循环。

- 我们以输入序列元素 2（即 $x^{(2)}$）为例来计算上下文向量 $z^{(2)}$；随后在本节中，我们将把它推广到计算所有上下文向量。
- 第一步是计算未归一化的 attention scores，方法是计算查询 $x^{(2)}$ 与所有其他输入 token 的点积：

In [3]:
query = inputs[1]  # 2nd input token is the query

attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query) # dot product (transpose not necessary here since they are 1-dim vectors)

print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


- 旁注：点积本质上就是「把两个向量逐元素相乘再求和」的一种简写：

In [4]:
res = 0.

for idx, element in enumerate(inputs[0]):
    res += inputs[0][idx] * query[idx]

print(res)
print(torch.dot(inputs[0], query))

tensor(0.9544)
tensor(0.9544)


- **第 2 步：** 对未归一化的 attention scores（「omega」，$\omega$）进行归一化，使它们之和为 1
- 下面是一种简单的归一化方法，使未归一化的 attention scores 之和为 1（这是一种约定，便于解释，对训练稳定性也很重要）：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/09.webp" width="500px">

**Figure 3.9 — 第二步：softmax 归一化（scores → weights）**

```
第 1 步的结果（未归一化）:
  attn_scores_2 = [0.95, 1.55, 1.44, 0.75, 0.64, 1.00]

第 2 步: softmax 归一化 → 和为 1

  原始分数        softmax 后
  ─────────       ─────────────
  0.95   ──→   0.14   ← "your"     (14%)
  1.55   ──→   0.26   ← "journey"  (26%) ← 最高权重
  1.44   ──→   0.23   ← "starts"   (23%)
  0.75   ──→   0.11   ← "with"     (11%)
  0.64   ──→   0.10   ← "one"      (10%) ← 最低权重
  1.00   ──→   0.16   ← "step"     (16%)
  ─────────       ─────────────
  总和: 6.33      总和: 1.00 ✓
```

| 归一化方法 | 公式 | 特点 |
|---|---|---|
| 简单归一化 | score_i / sum(scores) | 有负数时不可用 |
| **softmax** | exp(score_i) / Σexp(score_j) | 放大差异，处理负数，最常用 |

> **为什么用 softmax 而非简单除法？** ① softmax 的指数函数放大了高分和低分的差异（1.55 vs 0.64 → 0.26 vs 0.10）；② 即使有负数 score 也能正常工作；③ 梯度性质好，适合反向传播。本章先展示简单归一化，后续代码切换为 softmax。

In [5]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()

print("Attention weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)


- 然而在实践中，更常见且推荐的做法是使用 softmax 函数进行归一化，它在处理极端值方面表现更好，并在训练期间具有更理想的梯度性质。
- 下面是一个朴素的 softmax 实现用于缩放，它同样会把向量元素归一化到和为 1：

In [6]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_2_naive = softmax_naive(attn_scores_2)

print("Attention weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


- 上面的朴素实现在输入值很大或很小时，可能因溢出（overflow）和下溢（underflow）问题而出现数值不稳定
- 因此，实践中推荐使用 PyTorch 的 softmax 实现，它经过了高度的性能优化：

In [7]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)

print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


- **第 3 步：** 计算上下文向量 $z^{(2)}$，方法是将嵌入的输入 token $x^{(i)}$ 与 attention weights 相乘，再对所得向量求和：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/10.webp" width="500px">

**Figure 3.10 — 第三步：加权求和得到上下文向量 z(2)**

```
attn_weights_2 = [0.14, 0.26, 0.23, 0.11, 0.10, 0.16]
                  ↑                       ↑
                  各 token 的注意力权重（和=1.0）

context_vec_2 = 0.14*x(1) + 0.26*x(2) + 0.23*x(3) + 0.11*x(4) + 0.10*x(5) + 0.16*x(6)

  x(2) 权重最高(0.26) → "journey" 对 z(2) 贡献最大
  x(5) 权重最低(0.10) → "one" 对 z(2) 贡献最小
  shape: (3,) ← 与输入嵌入同维度
```

> **上下文向量 z(i) 的含义**：融合了整个序列的信息，但按相关性加权。z(2) 不再只是 "journey" 的嵌入，而是「在当前句子语境下 journey 的含义」。

In [8]:
query = inputs[1] # 2nd input token is the query

context_vec_2 = torch.zeros(query.shape)
for i,x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i]*x_i

print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


&nbsp;
### 3.3.2 计算所有输入 token 的注意力权重

#### 推广到所有输入序列 token：

- 在上面，我们为输入 2 计算了 attention weights 和上下文向量（如下图高亮行所示）
- 接下来，我们将此计算推广，计算所有的 attention weights 和上下文向量

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/11.webp" width="400px">

- （请注意，本图中的数字被截断到小数点后两位以减少视觉杂乱；每行的值之和应为 1.0 即 100%；类似地，其他图中的数字也是截断后的）

**Figure 3.11 — 推广到所有 token：完整的 attention weights 矩阵**

```
之前: 只计算了 x(2) 的 attention weights（一行）
现在: 对所有 6 个 token 重复同样计算 → 得到 6x6 矩阵

              x(1)    x(2)    x(3)    x(4)    x(5)    x(6)
  x(1) query [ 0.21,   0.18,   0.18,   0.13,   0.11,   0.19 ]   每行和=1.0
  x(2) query [ 0.14,   0.26,   0.23,   0.11,   0.10,   0.16 ]   ← 之前算的这行
  x(3) query [ 0.14,   0.24,   0.23,   0.11,   0.10,   0.17 ]
  x(4) query [ 0.19,   0.22,   0.19,   0.14,   0.08,   0.18 ]
  x(5) query [ 0.20,   0.15,   0.15,   0.11,   0.19,   0.19 ]
  x(6) query [ 0.13,   0.27,   0.23,   0.11,   0.10,   0.16 ]

  矩阵含义: 第 i 行 = x(i) 对所有 token 的关注程度
  对角线: token 对自己的权重（通常较高）
```

> **从 1 行到 6 行**：之前手动为 x(2) 计算了一行，现在用矩阵乘法一次性算出全部 6 行。这就是「推广到所有 token」的含义。矩阵中每行和为 1.0（softmax 保证）。

- 在 self-attention 中，过程从计算 attention scores 开始，随后将其归一化得到总和为 1 的 attention weights
- 这些 attention weights 接着被用于通过对输入的加权求和来生成上下文向量

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/12.webp" width="400px">

**Figure 3.12 — 用矩阵乘法一次性计算所有 attention scores**

```
之前（逐个计算）:           现在（矩阵乘法）:

  query = inputs[1]           attn_scores = inputs @ inputs.T
  scores = inputs @ query
  → 只有 x(2) 的一行           → 直接得到 6x6 完整矩阵

inputs (6x3)  @  inputs.T (3x6)  =  attn_scores (6x6)
┌───────────┐    ┌─────────────┐    ┌──────────────────┐
│ x(1) 3维  │    │             │    │ score(1,1) ... score(1,6)│
│ x(2) 3维  │ ×  │ inputs 的   │ =  │ score(2,1) ... score(2,6)│
│ ...       │    │ 转置 (3x6)  │    │ ...                      │
│ x(6) 3维  │    │             │    │ score(6,1) ... score(6,6)│
└───────────┘    └─────────────┘    └──────────────────┘

  score(i,j) = x(i) · x(j)  ← 第 i 个和第 j 个 token 的点积
```

| 方式 | 操作 | 计算量 |
|---|---|---|
| 逐个计算 | for 循环，每次 `inputs[i] @ query` | O(n) 次向量点积 |
| **矩阵乘法** | `inputs @ inputs.T` | 1 次矩阵乘法，GPU 并行 |

> **关键优化**：`inputs @ inputs.T` 一步到位算出所有 token 两两之间的点积，充分利用 GPU 矩阵运算的并行能力。

- 将前面的**第 1 步**应用到所有成对元素上，计算未归一化的 attention score 矩阵：

In [9]:
attn_scores = torch.empty(6, 6)

for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)

print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


- 我们可以通过矩阵乘法（matrix multiplication）更高效地实现与上面相同的结果：

In [10]:
attn_scores = inputs @ inputs.T
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


- 与前面的**第 2 步**类似，我们对每一行进行归一化，使每行的值之和为 1：

In [11]:
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


- 快速验证每行的值确实之和为 1：

In [12]:
row_2_sum = sum([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
print("Row 2 sum:", row_2_sum)

print("All row sums:", attn_weights.sum(dim=-1))

Row 2 sum: 1.0
All row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


- 应用前面的**第 3 步**来计算所有的上下文向量：

In [13]:
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


- 作为合理性检查（sanity check），之前计算的上下文向量 $z^{(2)} = [0.4419, 0.6515, 0.5683]$ 可以在上面输出的第 2 行找到：

In [14]:
print("Previous 2nd context vector:", context_vec_2)

Previous 2nd context vector: tensor([0.4419, 0.6515, 0.5683])


&nbsp;
## 3.4 实现带有可训练权重的自注意力

- 下图从概念上说明了本节开发的 self-attention 机制如何融入本书和本章的整体叙事与结构

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/13.webp" width="400px">

**Figure 3.13 — Section 3.4 概览：从简单自注意力到可训练自注意力**

```
3.3 简单自注意力（无可训练权重）        3.4 可训练自注意力（引入 Q/K/V）

  inputs (6x3)                           inputs (6x3)
     ↓                                      ↓
  inputs @ inputs.T → scores            x @ Wq → Q (query)
     ↓                                   x @ Wk → K (key)
  softmax → weights                      x @ Wv → V (value)
     ↓                                      ↓
  weights @ inputs → context            softmax(Q·K^T) → weights
                                            ↓
                                         weights @ V → context
```

> **核心改进**：3.3 中输入向量同时充当 query/key/value（一个角色干三件事）。3.4 引入三个可学习的权重矩阵 Wq、Wk、Wv，让模型自己学习「用什么去问」「暴露什么信息」「贡献什么内容」。

&nbsp;
### 3.4.1 逐步计算注意力权重

- 在本节中，我们将实现原始 Transformer 架构、GPT 模型以及大多数其他主流 LLM 中使用的 self-attention 机制
- 这种 self-attention 机制也被称为「缩放点积注意力」（scaled dot-product attention）
- 整体思路与之前类似：
  - 我们希望计算上下文向量，作为针对某个特定输入元素的输入向量加权和
  - 为此，我们需要 attention weights
- 如你将看到的，与前面介绍的基本注意力机制相比，只有一些细微差别：
  - 最显著的区别是引入了在模型训练过程中会被更新的权重矩阵
  - 这些可训练的权重矩阵至关重要，使模型（具体地，模型内部的 attention 模块）能够学习产生「好的」上下文向量

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/14.webp" width="600px">

**Figure 3.14 — Q/K/V 变换：通过权重矩阵将输入投影为三种角色**

```
输入 token x(2) = [0.55, 0.87, 0.66]  (3 维)

         Wq (3x2)         Wk (3x2)         Wv (3x2)
         ┌────────┐       ┌────────┐       ┌────────┐
x(2) ──→ │ 投影   │ ──→  q(2) (2 维)
         │        │
    ────→│ 投影   │ ──→  k(2) (2 维)
         │        │
    ────→│ 投影   │ ──→  v(2) (2 维)
         └────────┘

  q(2) = x(2) @ Wq   ← Query: 「我在找什么样的信息？」
  k(2) = x(2) @ Wk   ← Key:   「我能提供什么样的信息？」
  v(2) = x(2) @ Wv   ← Value: 「我实际贡献的内容是什么？」
```

| 角色 | 权重矩阵 | 直觉比喻 | 作用 |
|---|---|---|---|
| **Query** | Wq | 搜索查询 | 决定「我要关注什么样的 token」 |
| **Key** | Wk | 文档标签 | 决定「我有什么信息可被关注」 |
| **Value** | Wv | 实际内容 | 决定「被关注时我贡献什么」 |

> **为什么要拆成三个角色？** 同一个 token 在不同角色下可能需要不同的表示。例如 "cat" 作为 Query 时可能去寻找动词（「cat 想干什么」），作为 Key 时暴露自己的名词属性，作为 Value 时贡献具体的语义内容。三个权重矩阵让模型灵活学习这些不同的投影。

- 在逐步实现 self-attention 机制时，我们将首先引入三个训练权重矩阵 $W_q$、$W_k$ 和 $W_v$
- 这三个矩阵用于通过矩阵乘法将嵌入的输入 token $x^{(i)}$ 投影为 query、key 和 value 向量：

  - Query 向量：$q^{(i)} = x^{(i)}\,W_q $
  - Key 向量：$k^{(i)} = x^{(i)}\,W_k $
  - Value 向量：$v^{(i)} = x^{(i)}\,W_v $

> **为什么需要 Query / Key / Value 三组向量？**
>
> 上一节（3.3.1）的「简单注意力」直接用输入向量本身同时充当查询、被查询的内容和被聚合的内容。
> 但这样模型无法学习「用什么去匹配」和「匹配什么」以及「贡献什么」。引入三组可训练投影后：
>
> | 角色 | 含义 | 直觉类比 |
> |------|------|----------|
> | **Query（查询）** $q$ | 「我在寻找什么」 | 像在数据库中输入的检索关键词 |
> | **Key（键）** $k$ | 「我能提供什么，用于被匹配」 | 像数据库中每条记录的索引标签 |
> | **Value（值）** $v$ | 「一旦匹配上，我实际贡献的内容」 | 像数据库中记录的实际内容 |
>
> 计算流程：用 Query 与所有 Key 做点积 → softmax 得到权重 → 用权重对所有 Value 加权求和。
>
> **简单注意力 vs 带可训练权重的注意力**
>
> | 对比项 | 3.3 简单注意力 | 3.4 可训练注意力 (Q/K/V) |
> |--------|----------------|--------------------------|
> | 查询/键/值来源 | 都是 $x$ 本身 | 分别由 $x W_q, x W_k, x W_v$ 投影得到 |
> | 可学习参数 | 无 | 三个权重矩阵 $W_q, W_k, W_v$ |
> | 表达能力 | 固定，无法适配任务 | 可学习，能学到「怎么关注」 |
> | 缩放 | 无 | 除以 $\sqrt{d_k}$（数值稳定） |

- 输入 $x$ 的 embedding 维度与查询向量 $q$ 的维度可以相同也可以不同，取决于模型设计和具体实现
- 在 GPT 模型中，输入和输出维度通常相同；但为了演示方便、便于跟踪计算过程，我们在这里选择不同的输入和输出维度：

In [15]:
x_2 = inputs[1] # second input element
d_in = inputs.shape[1] # the input embedding size, d=3
d_out = 2 # the output embedding size, d=2

- 下面我们初始化三个权重矩阵；注意我们设置 `requires_grad=False` 是为了减少输出中的杂乱以便演示，但如果我们要在模型训练中使用这些权重矩阵，就应设置 `requires_grad=True`，以便在模型训练期间更新这些矩阵

In [16]:
torch.manual_seed(123)

W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key   = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

- 接下来计算 query、key 和 value 向量：

In [17]:
query_2 = x_2 @ W_query # _2 because it's with respect to the 2nd input element
key_2 = x_2 @ W_key 
value_2 = x_2 @ W_value

print(query_2)

tensor([0.4306, 1.4551])


- 如下面所示，我们成功地将 6 个输入 token 从 3 维 embedding 空间投影到了 2 维 embedding 空间：

In [18]:
keys = inputs @ W_key 
values = inputs @ W_value

print("keys.shape:", keys.shape)
print("values.shape:", values.shape)

keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])


- 在下一步（**第 2 步**）中，我们通过计算查询向量与每个 key 向量的点积来计算未归一化的 attention scores：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/15.webp" width="600px">

**Figure 3.15 — 第 2 步：用 Q 和 K 计算注意力分数**

```
之前（3.3 简单版）:  scores = inputs @ inputs.T     ← x 直接点积
现在（3.4 可训练）:  scores = Q @ K.T               ← Q/K 投影后再点积

Q (6x2)  @  K.T (2x6)  =  attn_scores (6x6)

  score(i,j) = q(i) · k(j)
             = (x(i)@Wq) · (x(j)@Wk)

  含义: token i 的「查询」与 token j 的「标签」有多匹配
```

> **与 3.3 的区别**：不再是原始嵌入直接点积，而是先投影到 Q/K 空间再做点积。Wq 和 Wk 是可训练参数，模型通过学习它们来决定「什么样的匹配模式最重要」。

In [19]:
keys_2 = keys[1] # Python starts index at 0
attn_score_22 = query_2.dot(keys_2)
print(attn_score_22)

tensor(1.8524)


- 由于我们有 6 个输入，因此对于给定的查询向量，我们有 6 个 attention scores：

In [20]:
attn_scores_2 = query_2 @ keys.T # All attention scores for given query
print(attn_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/16.webp" width="600px">

**Figure 3.16 — 第 3 步：softmax 归一化 + 缩放（scaled dot-product）**

```
attn_scores (6x6)
    ↓
    ↓ 缩放: scores / sqrt(d_k)     ← d_k = key 维度
    ↓   防止点积值过大导致 softmax 梯度消失
    ↓
softmax(dim=-1)
    ↓
attn_weights (6x6)   ← 每行和为 1.0
```

| 步骤 | 公式 | 目的 |
|---|---|---|
| 缩放 | scores / sqrt(d_k) | 防止点积过大 → softmax 饱和 → 梯度消失 |
| 归一化 | softmax(dim=-1) | 转为概率分布（每行和=1） |

> **为什么需要缩放？** 当 d_k 较大时，Q·K^T 的值会变大，softmax 会进入饱和区（输出接近 one-hot），导致梯度趋近于零。除以 sqrt(d_k) 将方差控制在 1 附近，保持梯度健康。

- 接下来在**第 3 步**，我们使用前面用过的 softmax 函数计算 attention weights（归一化后和为 1 的 attention scores）
- 与之前不同的是，我们现在通过除以 embedding 维度的平方根 $\sqrt{d_k}$（即 `d_k**0.5`）来对 attention scores 进行缩放：

In [21]:
d_k = keys.shape[1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print(attn_weights_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/17.webp" width="600px">

**Figure 3.17 — 第 4 步：用 attention weights 和 V 计算上下文向量**

```
attn_weights (6x6)  @  V (6x2)  =  context_vec (6x2)

  context(i) = Σ_j  attn_weights[i,j] * v(j)

  含义: token i 的上下文向量 = 所有 token 的 Value 按注意力权重加权求和

  对比 3.3 简单版:
    3.3: context = weights @ inputs     ← 加权求和原始嵌入
    3.4: context = weights @ V           ← 加权求和投影后的 Value
```

> **完整流程回顾**：x → (Wq/Wk/Wv) → Q/K/V → Q·K^T → 缩放 → softmax → weights → weights·V → context。这就是带可训练权重的自注意力，也是 Transformer 的核心计算单元。

- 在**第 4 步**，我们现在为输入查询向量 2 计算上下文向量：

In [22]:
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

tensor([0.3061, 0.8210])


&nbsp;
### 3.4.2 实现一个紧凑的 SelfAttention 类

> **类层级架构说明**
>
> 本节把前面零散的四步（投影 Q/K/V → 点积 → softmax 缩放 → 加权求和）封装进一个 `nn.Module`。
> `SelfAttention_v1` 用 `nn.Parameter(torch.rand(...))` 手动定义权重；`SelfAttention_v2` 改用 `nn.Linear`，
> 享受其更优的权重初始化方案，使训练更稳定。两个版本的 `forward` 计算流程完全一致。

- 把它们组合起来，我们可以这样实现 self-attention 机制：

In [23]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):

    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value
        
        attn_scores = queries @ keys.T # omega
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/18.webp" width="400px">

**Figure 3.18 — SelfAttention_v1 vs SelfAttention_v2 对比**

```
v1: 手动创建 Wq/Wk/Wv (nn.Parameter(torch.rand(d_in, d_out)))
    → 需要 x @ W（手动矩阵乘法）

v2: 使用 nn.Linear(d_in, d_out, bias=False)
    → 直接 self.W_query(x)，Linear 内部处理转置
```

| | SelfAttention_v1 | SelfAttention_v2 |
|---|---|---|
| **权重创建** | nn.Parameter(torch.rand(...)) | nn.Linear(bias=False) |
| **前向传播** | 手动 x @ W | self.Wq(x) |
| **性能** | 较慢（需要手动转置） | 较快（Linear 优化过） |
| **数值** | 结果略有不同 | 结果略有不同 |

> **注意**：v1 和 v2 输出不同，因为权重初始化的随机种子不同。两者数学上等价，但 nn.Linear 更高效且是 PyTorch 标准做法。

- 我们可以使用 PyTorch 的 Linear 层来简化上面的实现，在禁用偏置单元（bias）的情况下，Linear 层等价于矩阵乘法
- 使用 `nn.Linear` 相比我们手动的 `nn.Parameter(torch.rand(...))` 方式的另一大优势是：`nn.Linear` 拥有首选的权重初始化方案，能使模型训练更稳定

In [24]:
class SelfAttention_v2(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


- 注意，`SelfAttention_v1` 和 `SelfAttention_v2` 会给出不同的输出，因为它们为权重矩阵使用了不同的初始权重

&nbsp;
## 3.5 用因果注意力隐藏未来的词

- 在 causal attention（因果注意力）中，对角线以上的 attention weights 会被掩码（masked），确保对于任意给定输入，LLM 在用 attention weights 计算上下文向量时无法利用未来的 token

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/19.webp" width="400px">

**Figure 3.19 — 因果注意力（Causal Attention）概念**

```
普通自注意力:                    因果注意力:

  所有 token 互相可见              只能看自己和之前的 token

  x1 ←→ x2 ←→ x3 ←→ x4          x1 ←   x2 ←   x3 ←   x4
  ↑      ↑      ↑      ↑          ↑      ↑      ↑      ↑
  全部双向连接                    单向：只向左看

  attn_weights 矩阵:              attn_weights 矩阵:
  ┌──────────────────┐           ┌──────────────────┐
  │ ✓  ✓  ✓  ✓       │           │ ✓  ✗  ✗  ✗       │
  │ ✓  ✓  ✓  ✓       │           │ ✓  ✓  ✗  ✗       │  ← 上三角归零
  │ ✓  ✓  ✓  ✓       │           │ ✓  ✓  ✓  ✗       │
  │ ✓  ✓  ✓  ✓       │           │ ✓  ✓  ✓  ✓       │
  └──────────────────┘           └──────────────────┘
```

> **为什么需要因果注意力？** LLM 生成文本时是逐词生成的。训练时如果允许 token 「看到未来」，模型会作弊（直接抄答案）。因果 mask 确保预测第 i 个 token 时只能用 1~i 的信息。

&nbsp;
### 3.5.1 应用因果注意力掩码

- 在本节中，我们将前面的 self-attention 机制转换为 causal self-attention 机制
- Causal self-attention 确保模型对序列中某个位置的预测只依赖于已知的前序位置的输出，而不依赖未来位置
- 简而言之，这确保了每个下一个词的预测只依赖于前面的词
- 为此，对于每个给定 token，我们屏蔽掉（mask out）未来的 token（即在输入文本中位于当前 token 之后的那些 token）：

> **为什么需要因果掩码（causal mask）？**
>
> LLM 的训练目标是「根据前文预测下一个词」（next-token prediction）。如果在训练时让当前位置
> 「偷看」到未来的 token，模型就会作弊——测试时（生成时）未来词根本还不存在，这种信息泄露会导致：
>
> 1. 训练与推理不一致（训练能看到答案，推理看不到），模型学不到真正有用的表示；
> 2. 模型可能走捷径，直接复制未来信息，而非学习语言规律。
>
> 因果掩码用一个**下三角矩阵**把上三角（未来位置）置为 −∞，softmax 后这些位置权重变为 0，
> 从而保证每个位置只能关注自身及之前的 token。掩码矩阵示意（1=可见，0=屏蔽）：
>
> ```
>        t1  t2  t3  t4  t5  t6
>  t1  [  1   0   0   0   0   0 ]
>  t2  [  1   1   0   0   0   0 ]
>  t3  [  1   1   1   0   0   0 ]
>  t4  [  1   1   1   1   0   0 ]
>  t5  [  1   1   1   1   1   0 ]
>  t6  [  1   1   1   1   1   1 ]
> ```
> 这正是 `torch.tril`（下三角）生成的掩码。

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/20.webp" width="600px">

**Figure 3.20 — 因果 mask 的实现过程**

```
第 1 步: 计算 attention scores (6x6)
第 2 步: softmax → attention weights (6x6)
第 3 步: 创建上三角 mask，将对角线以上的权重置为 0
第 4 步: 重新归一化（每行重新除以行和）→ 行和恢复为 1.0

  原始 weights          mask 后              归一化后
  ┌────────────┐       ┌────────────┐       ┌────────────┐
  │ .21 .18 .18│       │ .21  0   0 │       │ 1.0  0   0 │
  │ .14 .26 .23│  ×    │ .14 .26  0 │  →    │ .35 .65  0 │
  │ .14 .24 .23│       │ .14 .24 .23│       │ .23 .40 .37│
  └────────────┘       └────────────┘       └────────────┘
```

> **mask + 归一化的两步法**：先置零再重新归一化，确保未被 mask 的权重仍然和为 1。

- 为了演示和实现 causal self-attention，让我们使用前一节的 attention scores 和 attention weights：

In [25]:
# Reuse the query and key weight matrices of the
# SelfAttention_v2 object from the previous section for convenience
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs) 
attn_scores = queries @ keys.T

attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


- 屏蔽未来 attention weights 最简单的方法是通过 PyTorch 的 `tril` 函数创建一个掩码，其中主对角线以下（含对角线本身）的元素设为 1，主对角线以上的元素设为 0：

In [26]:
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


- 然后，我们可以将 attention weights 与这个掩码相乘，把对角线以上的 attention scores 置零：

In [27]:
masked_simple = attn_weights*mask_simple
print(masked_simple)

tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)


- 然而，如果像上面那样在 softmax 之后再应用掩码，会破坏 softmax 所创建的概率分布
- softmax 确保所有输出值之和为 1
- 在 softmax 之后掩码就需要再次对输出重新归一化到和为 1，这使过程复杂化，并可能产生非预期效果

- 为了确保每行和为 1，我们可以这样归一化 attention weights：

In [28]:
row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)


- 虽然我们在编码 causal attention 机制上技术上已经完成，但让我们简要看看一种更高效的方法来实现上述效果
- 因此，与其把对角线以上的 attention weights 置零再重新归一化，不如在未归一化的 attention scores 进入 softmax 函数之前，把对角线以上的值用负无穷掩码：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/21.webp" width="450px">

**Figure 3.21 — 更高效的因果 mask：softmax 前置零**

```
方法 A（之前的做法）:                    方法 B（更高效）:
  scores → softmax → weights             scores → mask(-inf) → softmax
  → mask 上三角为 0                      → weights
  → 重新归一化

  3 步，且重新归一化有精度损失            2 步，softmax 自动处理 -inf

方法 B 的 scores 矩阵:
  ┌──────────────────────┐
  │  0.21   -inf  -inf   │     softmax(0.21, -inf, -inf)
  │  0.14   0.26  -inf   │ →   = (1.0, 0, 0)   ← -inf → 0
  │  0.14   0.24  0.23   │     
  └──────────────────────┘
```

> **为什么方法 B 更好？** ① 少一步重新归一化；② softmax(-inf) = 0 是精确的，没有精度损失；③ 是实际 GPT/Transformer 实现中使用的标准方法。

In [29]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)


- 如下面所示，现在每行的 attention weights 又正确地之和为 1：

In [30]:
attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


&nbsp;
### 3.5.2 用 dropout 掩码额外的注意力权重

> **什么是 Dropout，为什么放在 attention 之后？**
>
> Dropout（随机失活）在训练时随机把部分神经元的输出置零，迫使模型不依赖任何单一连接，从而**减少过拟合**。
> 这里我们把 dropout 施加在已归一化的 attention weights 上——即随机丢弃一部分「关注关系」，
> 让模型学到更鲁棒、不依赖某几条强关注的表示。注意 dropout **只在训练时生效**，推理时自动关闭。

- 此外，我们还应用 dropout 来减少训练期间的过拟合
- Dropout 可以应用在多个位置：
  - 例如，在计算完 attention weights 之后；
  - 或者在 attention weights 与 value 向量相乘之后
- 在这里，我们将在计算完 attention weights 之后应用 dropout 掩码，因为这种做法更常见

- 此外，在这个具体示例中，我们使用 50% 的 dropout 比率，意味着随机屏蔽掉一半的 attention weights。（之后在训练 GPT 模型时，我们将使用更低的 dropout 比率，例如 0.1 或 0.2）

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/22.webp" width="400px">

**Figure 3.22 — Dropout 在注意力中的应用**

```
attention weights (6x6)          dropout(0.5) 后
┌──────────────┐                ┌──────────────┐
│ 0.21  0.18   │                │ 0.42  0.00   │  ← 50% 置零
│ 0.14  0.26   │   ──→          │ 0.00  0.52   │  ← 剩余值 ×2
│ ...          │                │ ...          │
└──────────────┘                └──────────────┘

未被丢弃的值缩放因子 = 1 / (1 - dropout_rate)
  dropout_rate = 0.5 → 缩放 2x
  dropout_rate = 0.1 → 缩放 1.11x

目的: 训练时随机丢弃 50% 的注意力连接
      → 模型不能过度依赖任何单个连接
      → 提高泛化能力
```

> **注意**：dropout 只在训练时生效（`model.train()`）。推理时所有连接都保留（`model.eval()`）。这是防止过拟合的标准正则化技术。

- 如果我们应用 0.5（50%）的 dropout 比率，未被丢弃的值会相应地按 1/0.5 = 2 的因子进行缩放
- 缩放由公式 1 / (1 - `dropout_rate`) 计算

In [31]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5) # dropout rate of 50%
example = torch.ones(6, 6) # create a matrix of ones

print(dropout(example))

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


In [32]:
torch.manual_seed(123)
print(dropout(attn_weights))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.7599, 0.6194, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4921, 0.4925, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3966, 0.0000, 0.3775, 0.0000, 0.0000],
        [0.0000, 0.3327, 0.3331, 0.3084, 0.3331, 0.0000]],
       grad_fn=<MulBackward0>)


- 注意，得到的 dropout 输出可能因操作系统不同而看起来不同；你可以在[PyTorch issue 跟踪器](https://github.com/pytorch/pytorch/issues/121595)上阅读关于这种不一致的更多信息

&nbsp;
### 3.5.3 实现一个紧凑的因果自注意力类

- 现在，我们已经准备好实现一个可用的 self-attention，包括因果掩码和 dropout 掩码
- 还有一件事，就是要实现代码来处理包含多个输入的 batch，使我们的 `CausalAttention` 类支持我们在第 2 章实现的数据加载器所产生的 batch 输出
- 为简单起见，为了模拟这种 batch 输入，我们复制输入文本示例：

In [33]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape) # 2 inputs with 6 tokens each, and each token has embedding dimension 3

torch.Size([2, 6, 3])


In [34]:
class CausalAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length,
                 dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout) # New
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1)) # New

    def forward(self, x):
        b, num_tokens, d_in = x.shape # New batch dimension b
        # For inputs where `num_tokens` exceeds `context_length`, this will result in errors
        # in the mask creation further below.
        # In practice, this is not a problem since the LLM (chapters 4-7) ensures that inputs  
        # do not exceed `context_length` before reaching this forward method. 
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2) # Changed transpose
        attn_scores.masked_fill_(  # New, _ ops are in-place
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)  # `:num_tokens` to account for cases where the number of tokens in the batch is smaller than the supported context_size
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights) # New

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)

context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)

context_vecs = ca(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


- 注意，dropout 只在训练期间应用，在推理期间不应用

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/23.webp" width="500px">

**Figure 3.23 — CausalAttention 类的完整数据流**

```
inputs (batch, seq_len, d_in)
  ↓
  ├─ Wq → Q    ├─ Wk → K    ├─ Wv → V
  │
  ↓
Q @ K.T → scores → /sqrt(d_k) → softmax → weights
  │
  ↓
causal mask (上三角 → -inf)
  │
  ↓
dropout
  │
  ↓
weights @ V → context_vec (batch, seq_len, d_out)
```

> **完整因果注意力 = Q/K/V 投影 + 缩放点积 + 因果 mask + dropout**。这就是 GPT 中单个注意力头的完整实现。

&nbsp;
## 3.6 从单头注意力扩展到多头注意力

&nbsp;
### 3.6.1 堆叠多个单头注意力层

- 下面是之前实现的 self-attention 的总结（为简洁起见，未展示 causal 和 dropout 掩码）

- 这也被称为单头注意力（single-head attention）：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/24.webp" width="400px">

- 我们只需堆叠多个单头注意力模块，即可得到一个多头注意力（multi-head attention）模块：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/25.webp" width="400px">

> **为什么需要多个注意力头？**
>
> 单头注意力只能学到一种「关注模式」。但语言中的关系是多维度的：一个词可能同时
> 在关注语法依赖（主谓一致）、语义关系（同义/反义）、共指（指代消解）等不同方面。
> 多头注意力让模型并行运行多组注意力，每组用不同的、学习到的线性投影，> 从而**联合关注来自不同表示子空间、不同位置的信息**。最后把各头输出拼接，得到更丰富的表示。
>
> 形象地说：就像一组专家，每位专家从不同角度审视同一段文本，最后汇总各自的观点。

**Figure 3.24 — 单头注意力 vs 多头注意力**

```
单头注意力 (1 head):
  inputs → Wq/Wk/Wv → Q/K/V → attention → context
  output_dim = d_out (例如 256)

多头注意力 (num_heads 个头并行):
  inputs → 拆分为 num_heads 组 Wq/Wk/Wv
          head_0: Q0/K0/V0 (d_out/num_heads 维)
          head_1: Q1/K1/V1 (d_out/num_heads 维)
          ...
  每个头独立计算 attention
  → 拼接所有头的输出 → out_proj → context
  output_dim = d_out (不变)

例如: d_out=256, num_heads=8
  每个头: 256/8 = 32 维
  8 个头各关注不同模式: 语法、语义、位置、情感...
```

> **为什么多头？** 单头注意力只能学一种关注模式。多头让模型同时关注不同方面的信息：头 1 可能关注语法关系，头 2 关注指代消解，头 3 关注语义相似性。head_dim = d_out / num_heads 保证总维度不变。

In [35]:
class MultiHeadAttentionWrapper(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias) 
             for _ in range(num_heads)]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)


torch.manual_seed(123)

context_length = batch.shape[1] # This is the number of tokens
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(
    d_in, d_out, context_length, 0.0, num_heads=2
)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])


- 在上面的实现中，embedding 维度是 4，因为我们用 `d_out=2` 作为 key、query 和 value 向量以及上下文向量的 embedding 维度。并且由于我们有 2 个 attention head，所以输出 embedding 维度为 2*2=4

&nbsp;
### 3.6.2 用权重拆分实现多头注意力

- 虽然上面是一种直观且功能完整的 multi-head attention 实现（封装了前面的单头注意力 `CausalAttention` 实现），但我们可以编写一个名为 `MultiHeadAttention` 的独立类来实现同样的功能

- 对于这个独立的 `MultiHeadAttention` 类，我们不是拼接单个 attention head
- 相反，我们创建单个 W_query、W_key 和 W_value 权重矩阵，然后将它们拆分为每个 attention head 的独立矩阵：

> **多头并行计算与 head_dim**
>
> 与其创建 `num_heads` 个独立的 `CausalAttention` 对象（各自带一整套权重），这种实现只创建
> **一组**大的 $W_q, W_k, W_v$（输出维度为 `d_out`），然后用 `view` + `transpose` 把它**逻辑拆分**成多个头：
>
> ```
>  head_dim = d_out // num_heads
> ```
>
> 例如 `d_out=4, num_heads=2` → 每个 head 维度为 2。各 head 共享同一组大矩阵的参数，
> 但关注不同的子空间。张量形状变化：
>
> ```
>  (b, num_tokens, d_out)
>      → view → (b, num_tokens, num_heads, head_dim)
>      → transpose → (b, num_heads, num_tokens, head_dim)
> ```
>
> 转置后，每个 head 成为独立的 (num_tokens, head_dim) 矩阵，并行做点积注意力，
> 最后再转置回来并 `view` 合并为 (b, num_tokens, d_out)。这样一次矩阵乘法就完成所有 head 的计算，高效且节省显存。

In [36]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        # As in `CausalAttention`, for inputs where `num_tokens` exceeds `context_length`, 
        # this will result in errors in the mask creation further below. 
        # In practice, this is not a problem since the LLM (chapters 4-7) ensures that inputs  
        # do not exceed `context_length` before reaching this forward method.

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) 
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2) 
        
        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

torch.manual_seed(123)

batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


- 注意，上面本质上是 `MultiHeadAttentionWrapper` 的一个更高效的改写版本
- 由于随机权重初始化不同，输出看起来有些不同，但两者都是功能完整的实现，可用于我们在后续章节中实现的 GPT 类

---

**关于输出维度的说明**

- 在上面的 `MultiHeadAttention` 中，我使用 `d_out=2` 以便与前面的 `MultiHeadAttentionWrapper` 类保持相同设置
- `MultiHeadAttentionWrapper` 由于拼接，返回的输出头维度为 `d_out * num_heads`（即 `2*2 = 4`）
- 然而，`MultiHeadAttention` 类（为使其更易用）允许我们通过 `d_out` 直接控制输出头维度；这意味着，如果我们设置 `d_out = 2`，无论 head 数量多少，输出头维度都将是 2
- 事后看来，正如读者[指出](https://github.com/rasbt/LLMs-from-scratch/pull/859)的那样，对 `MultiHeadAttention` 使用 `d_out = 4` 可能更直观，这样它就能产生与 `d_out = 2` 的 `MultiHeadAttentionWrapper` 相同的输出维度。

---

- 另外请注意，我们在上面的 `MultiHeadAttention` 类中添加了一个线性投影层（`self.out_proj`）。这只是一个不改变维度的线性变换。在 LLM 实现中使用这样的投影层是一种标准约定，但并非严格必要（最近的研究表明它可以被移除而不影响建模性能；参见本章末尾的进一步阅读部分）

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/26.webp" width="400px">

**Figure 3.26 — MultiHeadAttention 的输出维度说明**

```
输入:  inputs (batch, seq_len, d_in)
输出:  context (batch, seq_len, d_out)

内部流程:
  qkv_weights (d_in, 3*d_out)
    → q, k, v 各 (batch, seq_len, d_out)
    → reshape 为 (batch, seq_len, num_heads, head_dim)
    → 转置为 (batch, num_heads, seq_len, head_dim)
    → 每个头独立 attention
    → 拼接回 (batch, seq_len, d_out)
    → out_proj (d_out, d_out) → 最终输出

  head_dim = d_out // num_heads
  例: d_out=256, num_heads=8 → head_dim=32
```

> **out_proj 的作用**：拼接后的多头输出经过一个线性层，让不同头的信息互相融合。虽然研究表明它不是严格必要的，但它是 GPT/Transformer 的标准组件。

- 注意，如果你对上述实现的一个紧凑且高效的版本感兴趣，也可以考虑 PyTorch 中的[`torch.nn.MultiheadAttention`](https://pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html) 类

- 由于上面的实现乍一看可能有点复杂，让我们看看执行 `attn_scores = queries @ keys.transpose(2, 3)` 时发生了什么：

In [37]:
# (b, num_heads, num_tokens, head_dim) = (1, 2, 3, 4)
a = torch.tensor([[[[0.2745, 0.6584, 0.2775, 0.8573],
                    [0.8993, 0.0390, 0.9268, 0.7388],
                    [0.7179, 0.7058, 0.9156, 0.4340]],

                   [[0.0772, 0.3565, 0.1479, 0.5331],
                    [0.4066, 0.2318, 0.4545, 0.9737],
                    [0.4606, 0.5159, 0.4220, 0.5786]]]])

print(a @ a.transpose(2, 3))

tensor([[[[1.3208, 1.1631, 1.2879],
          [1.1631, 2.2150, 1.8424],
          [1.2879, 1.8424, 2.0402]],

         [[0.4391, 0.7003, 0.5903],
          [0.7003, 1.3737, 1.0620],
          [0.5903, 1.0620, 0.9912]]]])


- 在这种情况下，PyTorch 的矩阵乘法实现会处理 4 维输入 tensor，使矩阵乘法在最后 2 个维度（num_tokens, head_dim）之间进行，然后对各个 head 重复执行

- 例如，下面是一种更紧凑的方式来分别计算每个 head 的矩阵乘法：

In [38]:
first_head = a[0, 0, :, :]
first_res = first_head @ first_head.T
print("First head:\n", first_res)

second_head = a[0, 1, :, :]
second_res = second_head @ second_head.T
print("\nSecond head:\n", second_res)

First head:
 tensor([[1.3208, 1.1631, 1.2879],
        [1.1631, 2.2150, 1.8424],
        [1.2879, 1.8424, 2.0402]])

Second head:
 tensor([[0.4391, 0.7003, 0.5903],
        [0.7003, 1.3737, 1.0620],
        [0.5903, 1.0620, 0.9912]])


&nbsp;
## 总结与要点

- 参见 [./multihead-attention.ipynb](./multihead-attention.ipynb) 代码 notebook，它是数据加载器（第 2 章）加上我们在本章实现并将在后续章节训练 GPT 模型时所需的 multi-head attention 类的精简版本
- 你可以在 [./exercise-solutions.ipynb](./exercise-solutions.ipynb) 中找到练习解答